# DRLB with linear lambda for fit + checkpoint lambda for inference

This notebook runs `LinearBidder` on the train split, estimates an implied DRLB lambda from `lambda = ctr_pred / bid`, uses it as `fit_lambda_init` for DRLB training, and starts inference from checkpoint-final lambda via `inference_lambda_init_mode=\"checkpoint_final\"`.


In [1]:
import sys
from dataclasses import replace
from pathlib import Path

import numpy as np
import pandas as pd

REPO_ROOT = "/Users/amsafin/code/local_ml/rl/bat-autobidding-benchmark"
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from example_notebooks.experiments.adapters.baseline_adapter import evaluate_baseline_model_inprocess
from example_notebooks.experiments.drlb.profiles import build_config as build_drlb_config
from example_notebooks.experiments.drlb.profiles import get_profile as get_drlb_profile
from example_notebooks.experiments.infra.split_utils import resolve_normalized_splits
from example_notebooks.experiments.shared_runner import run_experiment_inprocess
from simulator.model.linear_bidder import LinearBidder
from simulator.simulation.simulate import simulate_campaign
from simulator.validation.check_results import create_campaign_instance


/Users/amsafin/code/local_ml/rl/bat_venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
RUN_NAME = "drlb_with_linear_lambda_fit_checkpoint_infer_all"
DRLB_PROFILE = "drlb_smooth"
SPLIT_SET = "full_train_val_holdout"
N_TRIALS = 1
# True → train on all candidate timesteps (overrides profile default 64). Same idea as
# run_drlb_profile_inprocess(..., use_all_train_timesteps=True).
USE_ALL_TRAIN_STEPS = True
# When USE_ALL_TRAIN_STEPS is False: cap training steps, or leave None to keep the profile default (64 for drlb_smooth).
MAX_TRAIN_STEPS: int | None = None
VERBOSE = False
MEAN_CLICK_PRICE = 5.0

# Replace these with tuned linear params if you want the lambda prior to mirror a tuned baseline.
# LINEAR_PARAMS = {
#     "cold_start_coef": 0.023335213830958296,
#     "lower_clip": 9,
#     "upper_clip": 1,
#     "factor": 3.4224852046754637,
# }


In [3]:
import pickle

In [4]:
# Tuned linear params live under evaluate_baselines/best_params/<subfolder>/ (see baselines_finetune.BaseLineTrainer).
_linear_scr_fpa = Path(REPO_ROOT) / "example_notebooks" / "evaluate_baselines" / "best_params" / "fpa_baseline_n10_rndm_42" / "linear_scr_FPA.pkl"
with _linear_scr_fpa.open("rb") as f:
    linear_tuned_params = pickle.load(f)
# Last expression must be at module level — Jupyter does not auto-display values inside `with` / `if` / etc.
linear_tuned_params

{'coef': 0.023335213830958296,
 'lower_clip': 9,
 'upper_clip': 1,
 'factor': 3.4224852046754637}

In [5]:
LINEAR_PARAMS = linear_tuned_params

In [6]:
config = build_drlb_config(
    RUN_NAME,
    profile=DRLB_PROFILE,
    split_set=SPLIT_SET,
)
config = replace(config, n_trials=N_TRIALS)
if USE_ALL_TRAIN_STEPS:
    config = replace(config, max_steps=None)
elif MAX_TRAIN_STEPS is not None:
    config = replace(config, max_steps=int(MAX_TRAIN_STEPS))

normalized_splits = resolve_normalized_splits(config)
normalized_splits


{'train': {'campaigns_path': '/Users/amsafin/code/local_ml/rl/bat-autobidding-benchmark/data/fpa/campaigns_fpa_train_val.csv',
  'stats_path': '/Users/amsafin/code/local_ml/rl/bat-autobidding-benchmark/data/fpa/stats_fpa_train_val.csv'},
 'val': {'campaigns_path': '/Users/amsafin/code/local_ml/rl/bat-autobidding-benchmark/data/fpa/campaigns_fpa_val_val.csv',
  'stats_path': '/Users/amsafin/code/local_ml/rl/bat-autobidding-benchmark/data/fpa/stats_fpa_val_val.csv'},
 'test_holdout': {'campaigns_path': '/Users/amsafin/code/local_ml/rl/bat-autobidding-benchmark/data/fpa/campaigns_fpa_holdout_test.csv',
  'stats_path': '/Users/amsafin/code/local_ml/rl/bat-autobidding-benchmark/data/fpa/stats_fpa_holdout_test.csv'}}

In [7]:
train_campaigns = pd.read_csv(normalized_splits["train"]["campaigns_path"])
train_stats = pd.read_csv(normalized_splits["train"]["stats_path"])

linear_hist_parts = []
for _, campaign_row in train_campaigns.iterrows():
    campaign_id = int(campaign_row["campaign_id"])
    campaign_stats = train_stats[train_stats.campaign_id == campaign_id].copy()
    if campaign_stats.empty:
        continue

    campaign = create_campaign_instance(campaign_row, MEAN_CLICK_PRICE)
    bidder = LinearBidder(LINEAR_PARAMS)
    history = simulate_campaign(
        campaign=campaign,
        bidder=bidder,
        stats_file=campaign_stats,
        auction_mode=config.auction_mode,
    )
    hist_df = history.to_data_frame()
    if not hist_df.empty:
        linear_hist_parts.append(hist_df)

linear_hist = pd.concat(linear_hist_parts, ignore_index=True)
linear_hist.head()


,curr_time,curr_timestamp,campaign_start_time,campaign_end_time,campaign_id,balance,initial_balance,clicks,contacts,bid,loc_id,region_id,logical_category,microcat_ext,prev_timestamp,desired_clicks,desired_time,spend_history,clicks_history
0,1986-10-14 09:00:00,529653600,529652891,529739291,2978176,384.000000,384.0,0.000000,0.000000,115.200000,638790,637680,1.7,1087250,529650000,76.0,24,0.000000,0.000000
1,1986-10-14 10:00:00,529657200,529652891,529739291,2978176,378.692036,384.0,0.038640,0.002236,137.370552,638790,637680,1.7,1087250,529653600,76.0,24,5.307964,0.038640
2,1986-10-14 11:00:00,529660800,529652891,529739291,2978176,373.627122,384.0,0.069365,0.004013,164.844662,638790,637680,1.7,1087250,529657200,76.0,24,5.064913,0.030725
3,1986-10-14 12:00:00,529664400,529652891,529739291,2978176,304.715804,384.0,0.417730,0.033313,197.813595,638790,637680,1.7,1087250,529660800,76.0,24,68.911318,0.348365
4,1986-10-14 13:00:00,529668000,529652891,529739291,2978176,300.775216,384.0,0.453964,0.035409,108.754915,638790,637680,1.7,1087250,529664400,76.0,24,3.940588,0.036234


In [8]:
ctr_by_period = (
    train_stats
    .groupby(["campaign_id", "period"], as_index=False)
    .agg(ctr_pred=("CTRPredicts", "mean"))
)

lambda_df = linear_hist.merge(
    ctr_by_period,
    left_on=["campaign_id", "prev_timestamp"],
    right_on=["campaign_id", "period"],
    how="inner",
)
lambda_df = lambda_df[(lambda_df["bid"] > 0) & (lambda_df["ctr_pred"] > 0)].copy()
lambda_df["linear_lambda"] = lambda_df["ctr_pred"] / lambda_df["bid"]

linear_lambda_init = float(lambda_df["linear_lambda"].mean())
linear_lambda_median = float(lambda_df["linear_lambda"].median())

lambda_summary = lambda_df["linear_lambda"].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9])
lambda_summary, linear_lambda_init, linear_lambda_median


(count    3.108000e+04
 mean     2.842317e-03
 std      7.234237e-03
 min      7.047800e-07
 10%      9.220198e-05
 25%      2.600320e-04
 50%      8.196880e-04
 75%      2.204470e-03
 90%      6.242249e-03
 max      1.461533e-01
 Name: linear_lambda, dtype: float64,
 0.0028423174374845716,
 0.0008196880010539518)

In [9]:
lambda_df[["campaign_id", "prev_timestamp", "bid", "ctr_pred", "linear_lambda"]].head(20)


,campaign_id,prev_timestamp,bid,ctr_pred,linear_lambda
0,2978176,529650000,115.200000,0.031135,0.000270
1,2978176,529653600,137.370552,0.015867,0.000116
2,2978176,529657200,164.844662,0.024953,0.000151
3,2978176,529660800,197.813595,0.037718,0.000191
4,2978176,529664400,108.754915,0.024103,0.000222
5,2978176,529668000,137.370552,0.017334,0.000126
6,2978176,529671600,164.844662,0.048282,0.000293
7,2978176,529675200,197.813595,0.033528,0.000169
8,2978176,529678800,237.376314,0.041229,0.000174
9,2978176,529682400,284.851577,0.015753,0.000055


In [10]:
profile = get_drlb_profile(DRLB_PROFILE)
base_drlb_params = {
    **profile["base_drlb_params"],
    "fit_lambda_init": linear_lambda_init,
    "inference_lambda_init": None,
    "inference_lambda_init_mode": "checkpoint_final",
}

base_drlb_params


{'max_bid': 100.0,
 'T': 72,
 'lambda_min': 1e-06,
 'lambda_max': 10.0,
 'bids_per_timestep': 1,
 'dqn_soft_update_tau': 0.01,
 'dqn_loss_type': 'smooth_l1',
 'dqn_grad_clip_norm': 5.0,
 'dqn_reward_clip_value': 10.0,
 'reward_net_loss_type': 'smooth_l1',
 'reward_net_grad_clip_norm': 5.0,
 'reward_net_reward_clip_value': 10.0,
 'fit_lambda_init': 0.0028423174374845716,
 'inference_lambda_init': None,
 'inference_lambda_init_mode': 'checkpoint_final'}

In [11]:
result = run_experiment_inprocess(
    config,
    verbose=VERBOSE,
    base_drlb_params=base_drlb_params,
    reference_model_params=profile["reference_model_params"],
    state_type=profile["state_type"],
    objective=profile["objective"],
    search_space_fn=profile["search_space_fn"],
    n_trials=config.n_trials,
    max_train_steps=config.max_steps,
)

result["summary"]


[I 2026-05-03 00:11:26,131] A new study created in memory with name: no-name-7574ec3a-65be-4101-84bf-6aae88d576c3
[I 2026-05-03 00:14:06,377] Trial 0 finished with value: 1985.1032474644915 and parameters: {'dqn_gamma': 0.9779639145570413, 'dqn_lr': 0.0004746256651638466, 'dqn_target_update_interval': 170, 'reward_net_lr': 0.001916219096572235}. Best is trial 0 with value: 1985.1032474644915.


{'experiment_name': 'drlb_with_linear_lambda_fit_checkpoint_infer_all',
 'family': 'drlb',
 'run_name': 'drlb_with_linear_lambda_fit_checkpoint_infer_all',
 'auction_mode': 'FPA',
 'objective_metric': 'SCR',
 'objective_type': 'clicks',
 'split_set': 'full_train_val_holdout',
 'split_fingerprint': '3fd164e7f75d995c46b491cc445cdeaf7850e2e9e6518272ce817b5ac5a7c1d1',
 'timestamp': '2026-05-02T21:22:14+00:00',
 'git_hash': '1d3848f',
 'data_splits': {'train': {'campaigns_path': '/Users/amsafin/code/local_ml/rl/bat-autobidding-benchmark/data/fpa/campaigns_fpa_train_val.csv',
   'stats_path': '/Users/amsafin/code/local_ml/rl/bat-autobidding-benchmark/data/fpa/stats_fpa_train_val.csv'},
  'val': {'campaigns_path': '/Users/amsafin/code/local_ml/rl/bat-autobidding-benchmark/data/fpa/campaigns_fpa_val_val.csv',
   'stats_path': '/Users/amsafin/code/local_ml/rl/bat-autobidding-benchmark/data/fpa/stats_fpa_val_val.csv'},
  'test_holdout': {'campaigns_path': '/Users/amsafin/code/local_ml/rl/bat-aut

In [12]:
result["best_run"]["metrics"]


{'cpc_relative': 333.35253639453543,
 'rmse': 2.1382605302657898,
 'clicks_sum': 10376.893900096995,
 'quickspend': 0.16653696498054474,
 'skipped_campaigns': 0,
 'time_inference_sec': 70.5913028717041,
 'time_overall_sec': 93.03509306907654,
 'average_end_balance_share': 0.28694598981609915,
 'label': 'best_refit',
 'train_steps': 48240,
 'last_dqn_loss': 10.891658782958984,
 'last_reward_net_loss': 1.245486855506897,
 'dqn_loss_mean': 5.766258592044296,
 'dqn_loss_p95': 10.27133936882019,
 'reward_net_loss_mean': 1.5913778071292963,
 'reward_net_loss_p95': 2.3236397743225097,
 'reward_signal_mean': 3.9522903656174626,
 'lambda_final': 0.0015726745056395938}

In [13]:
bidder = result["best_run"]["bidder"]
diagnostics = bidder.get_training_diagnostics()
diagnostics.tail()


,global_t,rem_budget,lambda,eps,dqn_action,dqn_loss,reward_signal,reward_net_loss
48235,48236,233,0.001817,0.05,1,4.019251,5.886324,1.663796
48236,48237,233,0.001671,0.05,0,8.454774,4.655035,2.238356
48237,48238,233,0.001621,0.05,1,7.949540,3.966651,1.417162
48238,48239,233,0.001621,0.05,3,15.400972,2.591113,1.334667
48239,48240,233,0.001573,0.05,1,10.891659,1.320776,1.245487


In [14]:
drlb_metrics = result["best_run"]["metrics"]
linear_holdout_run = evaluate_baseline_model_inprocess(
    model_name="linear",
    label="linear_holdout",
    params_dict={
        "coef": LINEAR_PARAMS["coef"],
        "lower_clip": LINEAR_PARAMS["lower_clip"],
        "upper_clip": LINEAR_PARAMS["upper_clip"],
        "factor": LINEAR_PARAMS["factor"],
    },
    split=normalized_splits["test_holdout"],
    auction_mode=config.auction_mode,
)
linear_metrics = linear_holdout_run["metrics"]
pd.DataFrame(
    [
        {"model": "linear", **linear_metrics},
        {"model": "drlb_linear_fit_checkpoint_infer", **drlb_metrics},
    ]
)[["model", "clicks_sum", "cpc_relative", "rmse", "quickspend", "time_inference_sec"]]


,model,clicks_sum,cpc_relative,rmse,quickspend,time_inference_sec
0,linear,17792.733546,421.338042,1.503115,0.004669,96.576326
1,drlb_linear_fit_checkpoint_infer,10376.893900,333.352536,2.138261,0.166537,70.591303


In [15]:
comparison_delta = pd.DataFrame(
    {
        "metric": ["clicks_sum", "cpc_relative", "rmse", "quickspend"],
        "linear": [
            linear_metrics["clicks_sum"],
            linear_metrics["cpc_relative"],
            linear_metrics["rmse"],
            linear_metrics["quickspend"],
        ],
        "drlb": [
            drlb_metrics["clicks_sum"],
            drlb_metrics["cpc_relative"],
            drlb_metrics["rmse"],
            drlb_metrics["quickspend"],
        ],
    }
)
comparison_delta["drlb_minus_linear"] = comparison_delta["drlb"] - comparison_delta["linear"]
comparison_delta


,metric,linear,drlb,drlb_minus_linear
0,clicks_sum,17792.733546,10376.893900,-7415.839646
1,cpc_relative,421.338042,333.352536,-87.985506
2,rmse,1.503115,2.138261,0.635145
3,quickspend,0.004669,0.166537,0.161868
